# Text Analysis of Customer Transactions

## Objective
To examine text-based fields in the transaction dataset, identify recurring patterns, and extract meaningful insights from transaction descriptions and merchant location information.

In [1]:
# cell 1

import pandas as pd
from pathlib import Path

# Load the cleaned transaction dataset
file_path = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "cleaned_transactions.csv"
)

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (12043, 24)


In [2]:
# cell 2

# Identify columns containing text data
text_columns = df.select_dtypes(include=["object", "str"]).columns.tolist()

print("Text columns:")
for column in text_columns:
    print(f"- {column}")

print("\nNumber of text columns:", len(text_columns))


Text columns:
- status
- bpay_biller_code
- account
- currency
- long_lat
- txn_description
- merchant_id
- first_name
- date_original
- gender
- merchant_suburb
- merchant_state
- extraction
- transaction_id
- country
- customer_id
- merchant_long_lat
- movement
- date

Number of text columns: 19


In [3]:
# cell 3

# Inspect selected text columns
columns_to_inspect = [
    "txn_description",
    "merchant_suburb",
    "merchant_state",
    "movement",
    "status"
]

for column in columns_to_inspect:
    print(f"\n--- {column} ---")
    print("Unique values:", df[column].nunique())
    print(df[column].value_counts().head(10))



--- txn_description ---
Unique values: 6
txn_description
SALES-POS     3934
POS           3783
PAYMENT       2600
PAY/SALARY     883
INTER BANK     742
PHONE BANK     101
Name: count, dtype: int64

--- merchant_suburb ---
Unique values: 1609
merchant_suburb
Melbourne        255
Sydney           233
Southport         82
Brisbane City     79
Chatswood         55
Perth             45
Adelaide          44
Mount Gambier     41
Broadbeach        41
Parramatta        39
Name: count, dtype: int64

--- merchant_state ---
Unique values: 8
merchant_state
NSW    2169
VIC    2131
QLD    1556
WA     1100
SA      415
NT      205
ACT      73
TAS      68
Name: count, dtype: int64

--- movement ---
Unique values: 2
movement
debit     11160
credit      883
Name: count, dtype: int64

--- status ---
Unique values: 2
status
authorized    7717
posted        4326
Name: count, dtype: int64


In [4]:
# cell 4

# Create cleaned copies of selected text columns

df["txn_description_clean"] = (
    df["txn_description"].str.strip().str.upper()
)

df["merchant_suburb_clean"] = (
    df["merchant_suburb"].str.strip().str.title()
)

df["merchant_state_clean"] = (
    df["merchant_state"].str.strip().str.upper()
)

# Compare original and cleaned unique values
columns_to_compare = {
    "txn_description": "txn_description_clean",
    "merchant_suburb": "merchant_suburb_clean",
    "merchant_state": "merchant_state_clean"
}

for original, cleaned in columns_to_compare.items():

    print(f"\n{original}")
    print("Original unique values:", df[original].nunique())
    print("Cleaned unique values:", df[cleaned].nunique())



txn_description
Original unique values: 6
Cleaned unique values: 6

merchant_suburb
Original unique values: 1609
Cleaned unique values: 1606

merchant_state
Original unique values: 8
Cleaned unique values: 8


In [5]:
# cell 5
# Find original suburb names that became identical after cleaning

suburb_variations = (
    df.groupby("merchant_suburb_clean")["merchant_suburb"]
    .unique()
)

for cleaned_name, original_names in suburb_variations.items():

    if len(original_names) > 1:
        print(f"Standardised name: {cleaned_name}")
        print("Original variations:", list(original_names))
        print()


Standardised name: Kensington
Original variations: ['KENSINGTON', 'Kensington']

Standardised name: Mermaid Waters
Original variations: ['Mermaid Waters', 'MERMAID WATERS']

Standardised name: Sydney
Original variations: ['Sydney', 'SYDNEY']



In [6]:
# cell 6

# Count transaction descriptions
description_counts = (
    df["txn_description_clean"]
    .value_counts()
    .reset_index()
)

description_counts.columns = [
    "Transaction Description",
    "Transaction Count"
]

# Calculate percentage contribution
description_counts["Percentage"] = (
    description_counts["Transaction Count"]
    / len(df) * 100
)

display(description_counts.round(2))


,Transaction Description,Transaction Count,Percentage
0,SALES-POS,3934,32.67
1,POS,3783,31.41
2,PAYMENT,2600,21.59
3,PAY/SALARY,883,7.33
4,INTER BANK,742,6.16
5,PHONE BANK,101,0.84


In [7]:
# cell 7

# Cross-tabulation of transaction descriptions and movement
description_movement = pd.crosstab(
    df["txn_description_clean"],
    df["movement"],
    margins=True
)

display(description_movement)


movement,credit,debit,All
txn_description_clean,,,
INTER BANK,0,742,742
PAY/SALARY,883,0,883
PAYMENT,0,2600,2600
PHONE BANK,0,101,101
POS,0,3783,3783
SALES-POS,0,3934,3934
All,883,11160,12043


In [8]:
# cell 8

# Find the 15 most frequent merchant suburbs
top_suburbs = (
    df["merchant_suburb_clean"]
    .value_counts()
    .head(15)
    .reset_index()
)

top_suburbs.columns = [
    "Merchant Suburb",
    "Transaction Count"
]

display(top_suburbs)


,Merchant Suburb,Transaction Count
0,Melbourne,255
1,Sydney,235
2,Southport,82
3,Brisbane City,79
4,Chatswood,55
5,Perth,45
6,Adelaide,44
7,Mount Gambier,41
8,Broadbeach,41
9,Parramatta,39


In [9]:
# cell 9

# Analyse merchant suburb distribution by movement

suburb_movement = pd.crosstab(
    df["merchant_suburb_clean"],
    df["movement"]
)

# Calculate total transactions per suburb
suburb_movement["Total Transactions"] = (
    suburb_movement.sum(axis=1)
)

# Display the 15 suburbs with the highest transaction activity
top_suburb_movement = (
    suburb_movement
    .sort_values(
        by="Total Transactions",
        ascending=False
    )
    .head(15)
)

display(top_suburb_movement)

movement,debit,Total Transactions
merchant_suburb_clean,,
Melbourne,255,255
Sydney,235,235
Southport,82,82
Brisbane City,79,79
Chatswood,55,55
Perth,45,45
Adelaide,44,44
Mount Gambier,41,41
Broadbeach,41,41


In [10]:
# cell 10

# Check merchant suburb availability by transaction description

location_availability = pd.crosstab(
    df["txn_description_clean"],
    df["merchant_suburb"].isna(),
    margins=True
)

location_availability.columns = [
    "Location Available",
    "Location Missing",
    "Total"
]

display(location_availability)


,Location Available,Location Missing,Total
txn_description_clean,,,
INTER BANK,0,742,742
PAY/SALARY,0,883,883
PAYMENT,0,2600,2600
PHONE BANK,0,101,101
POS,3783,0,3783
SALES-POS,3934,0,3934
All,7717,4326,12043


In [11]:
# cell 11

# Analyse transaction distribution across merchant states

state_counts = (
    df["merchant_state"]
    .dropna()
    .str.strip()
    .str.upper()
    .value_counts()
    .reset_index()
)

state_counts.columns = [
    "Merchant State",
    "Transaction Count"
]

state_counts["Percentage"] = (
    state_counts["Transaction Count"]
    / state_counts["Transaction Count"].sum()
    * 100
)

display(state_counts.round(2))

,Merchant State,Transaction Count,Percentage
0,NSW,2169,28.11
1,VIC,2131,27.61
2,QLD,1556,20.16
3,WA,1100,14.25
4,SA,415,5.38
5,NT,205,2.66
6,ACT,73,0.95
7,TAS,68,0.88


In [12]:
# cell 12
state_description = pd.crosstab(
    df["merchant_state"].str.strip().str.upper(),
    df["txn_description_clean"]
)

display(state_description)

txn_description_clean,POS,SALES-POS
merchant_state,,
ACT,30,43
NSW,1075,1094
NT,110,95
QLD,754,802
SA,203,212
TAS,26,42
VIC,1045,1086
WA,540,560


In [13]:
# cell 13
state_description_percentage = pd.crosstab(
    df["merchant_state"].str.strip().str.upper(),
    df["txn_description_clean"],
    normalize="index"
) * 100

display(state_description_percentage.round(2))

txn_description_clean,POS,SALES-POS
merchant_state,,
ACT,41.10,58.90
NSW,49.56,50.44
NT,53.66,46.34
QLD,48.46,51.54
SA,48.92,51.08
TAS,38.24,61.76
VIC,49.04,50.96
WA,49.09,50.91


## Conclusion

The text and categorical-field analysis revealed several useful patterns in customer transactions.

1. **Transaction descriptions:** The dataset contains six fixed transaction-description categories rather than free-form text. SALES-POS (32.67%) and POS (31.41%) are the two most frequent categories, together accounting for 64.08% of all transactions.

2. **Text standardisation:** Standardising transaction descriptions, merchant suburbs, and merchant states improved consistency. Suburb standardisation reduced the number of unique values from 1,609 to 1,606 by resolving three capitalization variations.

3. **Transaction movement:** All PAY/SALARY transactions are credits, while the remaining four transaction-description categories are debits. This confirms that transaction descriptions directly reveal the target variable, making them unsuitable as input features for the classification model because of target leakage.

4. **Merchant locations:** Melbourne (255 transactions) and Sydney (235 transactions) have the highest recorded merchant-suburb transaction counts.

5. **State distribution:** NSW accounts for the largest share of recorded merchant-state transactions (28.11%), followed by VIC (27.61%) and QLD (20.16%).

6. **Transaction-type distribution across states:** POS and SALES-POS transactions have relatively similar proportions across most states. ACT and TAS show a higher proportion of SALES-POS transactions, while NT has a higher proportion of POS transactions.

### Limitations

- The dataset contains fixed transaction labels rather than natural-language descriptions or customer comments. Therefore, advanced NLP techniques such as sentiment analysis and topic modelling are not appropriate for this dataset.
- Merchant location information is available for only 7,717 transactions (POS and SALES-POS). The remaining 4,326 transactions do not contain merchant-suburb information.
- Geographical findings therefore represent only transactions with recorded merchant locations, not the entire transaction dataset.

Overall, this analysis demonstrates how text standardisation, categorical text exploration, and merchant-location analysis can reveal transaction patterns and identify important limitations in the available data.